# Storytelling with Data: รายได้เฉลี่ยของครัวเรือน
รายได้เฉลี่ยต่อเดือนของครัวเรือน จำแนกตามแหล่งที่มาของรายได้ และสถานะทางเศรษฐสังคมของครัวเรือน

## 1️⃣ Setup & Load Data  
ติดตั้งและ Import Libraries

In [ ]:
!gdown --id 1lqNkEQhIG89y26i6kjgXkRxyGTpngE0e
!unzip -q dataset.zip

In [ ]:
# Import libraries หลัก
import pandas as pd
import numpy as np
import plotly.express as px

# Tip: Use this to make Matplotlib plots look high-resolution in Colab
%config InlineBackend.figure_format = 'retina'

print("✅ Import สำเร็จ!")

In [ ]:
# โหลด รายได้เฉลี่ยต่อเดือนของครัวเรือน dataset
df = pd.read_csv('avg_income.csv')
print(f"📊 Dataset shape: {df.shape}")
print(f"   - จำนวนตัวอย่าง (rows): {df.shape[0]}")
print(f"   - จำนวน features (columns): {df.shape[1]}")
df.head(5)

## 2️⃣ EDA เบื้องต้น




In [ ]:
# ดูข้อมูลเบื้องต้น
print("📋 ข้อมูลทั่วไป:")
print("=" * 50)
df.info()

# แสดงสรุปข้อมูลเบื้องต้น
print("\n📋 สรุปข้อมูลเบื้องต้น:")
unique_counts = pd.DataFrame({
    'Feature': df.columns,
    'Unique_Values': [df[col].nunique() for col in df.columns],
    'Data_Type': [df[col].dtype for col in df.columns],
    'Missing_Values (%)': (df.isnull().sum() / len(df) * 100).values
})

print(unique_counts.sort_values(by='Unique_Values', ascending=False))

In [ ]:
# นับจำนวนแถวที่มีข้อมูลซ้ำกัน
print(f"Exact duplicates: {df.duplicated().sum()}")

# แสดงแถวที่มีข้อมูลซ้ำกัน
if df.duplicated().any():
    display(df[df.duplicated(keep=False)].sort_values(by=list(df.columns)).head(10))
    # Drop them
    df = df.drop_duplicates().reset_index(drop=True)

## 3️⃣ Data Quality & Cleaning 🧹
ตรวจสอบข้อมูลผิดปกติ
ก่อนวิเคราะห์จริง เราต้องทำความสะอาดข้อมูลก่อน ขั้นตอนนี้ครอบคลุมการ

(1) Filter ข้อมูลรายได้มากกว่า 0 บาท

(2) ลบ column attribute เป็นการบอกรายละเอียดของ rows ที่มีรายได้เท่ากับ 0

(3) rename คอลัมน์ให้สอดคล้องกัน

Data Cleaning ที่ดีจะทำให้ผลการวิเคราะห์ถูกต้องและน่าเชื่อถือ

### "garbage in, garbage out" is the law.



### (1) Filter ข้อมูลรายได้มากกว่า 0 บาท

นำข้อมูลออกไป 11.31% ของข้อมูลทั้งหมด เพื่อเพิ่มความแม่นบำและสอดคล้องกับการนำข้อมูลรายไปใช้ในการพิจารณาต่อไป

In [ ]:
# แสดงแถวที่ลบออก โดยระบุเป็น % แบ่งตามจังหวัด
dropped_df = df[df['value'] <= 0]
print("การกระจายตัวของแถวที่ถูกลบออกแบ่งตามจังหวัด:")
print(dropped_df['province'].value_counts(normalize=True) * 100)

In [ ]:

# ระบุแถวที่ รายได้ <= 0
condition = df['value'] <= 0

# คำนวณจำนวนแถว
total_rows = len(df)
rows_to_drop = condition.sum()
percent_dropped = (rows_to_drop / total_rows) * 100

# แสดงข้อมูลจำนวนแถว
print(f"Total rows in dataset: {total_rows}")
print(f"Rows to be dropped:    {rows_to_drop}")
print(f"Percentage to drop:    {percent_dropped:.2f}%")

# ทำการเก็บแถวที่ condition ไม่เป็นจริง และ copy ข้อมูลไปยัง df_clean
df_clean = df[~condition].copy()

summary = pd.DataFrame({
    'Metric': ['Original Rows', 'Dropped Rows', 'Remaining Rows'],
    'Value': [total_rows, rows_to_drop, len(df_clean)],
    'Percentage': ['100%', f'{percent_dropped:.2f}%', f'{100 - percent_dropped:.2f}%']
})
display(summary)

### (2) ลบ attribute column
attirbute column สำหรับการบอกรายละเอียดของ rows ที่มีรายได้เท่ากับ 0 ซึ่งไม่มีการนำมาใช้พิจารณา insigths ข้อมูล

In [ ]:
df_clean = df_clean.drop(columns=['attribute'])  # Features

# แสดงสรุปข้อมูลเบื้องต้น
print("\n📋 สรุปข้อมูลเบื้องต้น:")
unique_counts = pd.DataFrame({
    'Feature': df_clean.columns,
    'Unique_Values': [df_clean[col].nunique() for col in df_clean.columns],
    'Data_Type': [df_clean[col].dtype for col in df_clean.columns],
    'Missing_Values (%)': (df_clean.isnull().sum() / len(df_clean) * 100).values
})

print(unique_counts.sort_values(by='Unique_Values', ascending=False))

### (3) rename คอลัมน์ให้สอดคล้องกัน

In [ ]:
# Rename 'value' for representation
df_clean = df_clean.rename(columns={'value': 'income_thb'})

### 4️⃣ Advanced EDA & Insights 🔬
**4.1 📊 ใครได้รับค่าตอบแทนแค่ไหน? (Who earns what?)**

เพื่อให้เห็นภาพรายได้ของครัวเรือน ซึ่งจำแนกตามกลุ่มอาชีพและประเภทผู้ประกอบการ
รวมทั้งการตอบโจทย์ "ช่องว่างที่ผิดปกติ" เราจะใช้ Box Plot ซึ่งเป็นกราฟที่ออกแบบมาเพื่อจับ Outliers และแสดงความกว้างของช่องว่างรายได้โดยเฉพาะ  


In [ ]:
# คำนวณค่าเฉลี่ยของแต่ละกลุ่มอาชีพย่อย เพื่อสร้าง Bar Chart
summary = df_clean.groupby(['soc_eco_class1', 'soc_eco_class2'])['income_thb'].mean().reset_index()
summary = summary.sort_values(by='income_thb', ascending=True)

# ---------------------------------------------------------
# กราฟที่ 1: Horizontal Bar Chart (เปรียบเทียบรายได้)
# ---------------------------------------------------------
fig1 = px.bar(
    summary,
    x='income_thb',
    y='soc_eco_class2',
    color='soc_eco_class1', # ใช้สีแยกประเภทสถานะการทำงาน (เช่น ลูกจ้าง, เกษตรกร)
    orientation='h',
    title='<b>เปรียบเทียบรายได้เฉลี่ยต่อเดือน: กลุ่มอาชีพใดทิ้งห่างกลุ่มอื่น?</b>',
    labels={'income_thb': 'รายได้เฉลี่ย (บาท/เดือน)', 'soc_eco_class2': 'กลุ่มอาชีพย่อย', 'soc_eco_class1': 'สถานะการทำงาน'},
    color_discrete_sequence=px.colors.qualitative.Safe,
    text_auto='.0f' # โชว์ตัวเลขบนกราฟ
)
fig1.update_layout(font=dict(size=14), legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1))
fig1.show()

# ---------------------------------------------------------
# กราฟที่ 2: Box Plot (แสดงช่องว่างและความผิดปกติ)
# ---------------------------------------------------------
fig2 = px.box(
    df_clean,
    x='income_thb',
    y='soc_eco_class1',
    color='soc_eco_class1',
    points="outliers", # โชว์เฉพาะจุดที่ผิดปกติ (Outliers)
    title='<b>ช่องว่างรายได้และความเหลื่อมล้ำภายในแต่ละสถานะการทำงาน (Box Plot)</b>',
    labels={'income_thb': 'รายได้รวม (บาท/เดือน)', 'soc_eco_class1': 'สถานะการทำงาน'}
)
fig2.update_layout(font=dict(size=14), showlegend=False)
fig2.show()

จาก Visualization จะเห็นได้ว่า   
* ช่องว่างที่มีความห่างของรายได้อยู่ที่ "ลูกจ้าง"

  รายได้สูงที่สุด: คือกลุ่ม ผู้จัดการ นักวิชาการ และผู้ปฏิบัติงานวิชาชีพ (หมวดลูกจ้าง) นำโด่งอยู่ที่ประมาณ 48,292 บาท/เดือน

  รายได้ต่ำที่สุด: คือกลุ่ม คนงานเกษตร ป่าไม้ และประมง (หมวดลูกจ้างเช่นกัน) รั้งท้ายอยู่ที่ประมาณ 15,803 บาท/เดือน  
  
  💡 **Insight**: ช่องว่างรายได้ที่กว้างที่สุดไม่ได้เกิดระหว่าง 'นายจ้าง' กับ 'ลูกจ้าง' แต่เกิดขึ้น ภายในกลุ่มลูกจ้างด้วยกันเอง โดยลูกจ้างระดับบนมีรายได้ทิ้งห่างลูกจ้างระดับล่างถึงกว่า 3 เท่าตัว (ห่างกันเกิน 32,000 บาท) นี่คือช่องว่างที่ผิดปกติและสะท้อนความเหลื่อมล้ำของทักษะวิชาชีพ (Skill Gap)  

* กลุ่มผู้ประกอบธุรกิจของตนเองที่ไม่ใช่การเกษตร มีรายได้เฉลี่ยเกาะกลุ่มอยู่ที่ประมาณ 33,298 บาท/เดือน

  💡 **Insight**: กลุ่มนี้มีฐานรายได้ที่ค่อนข้างแข็งแรงและเกาะกลุ่มกัน ไม่ได้มีช่องว่างกว้างเหมือนกลุ่มลูกจ้าง

* ความผิดปกติในกลุ่ม "เกษตรกร"

  กลุ่มปลูกพืช/เลี้ยงสัตว์ ที่เช่าที่ดินหรือทำฟรี (29,401 บาท) กลับมีรายได้เฉลี่ยสูงกว่ากลุ่ม ที่เป็นเจ้าของที่ดินเอง (27,622 บาท) เล็กน้อย

  💡 **Insight**: เกษตรกรที่เช่าที่ดินกลับมีรายได้รวมสูงกว่าเจ้าของที่ดิน ซึ่งสมมติฐานทาง Data ชี้ว่า กลุ่มที่เช่าที่ดินอาจทำเกษตรกรรมเชิงพาณิชย์ขนาดใหญ่ (Commercial Farming) ในขณะที่เจ้าของที่ดินส่วนใหญ่อาจเป็นเกษตรกรรายย่อยแบบดั้งเดิม
  
🌵 Note: การที่ไม่ได้จัดการนำ outliers ออกจากการพิจารณา เนื่องจากข้อมูลทั้งหมดแสดงให้เห็นความขัดเจนในการนำเสนอตามเหตุผลที่กล่าวข้างต้น

**4.2 📊 ความเหลื่อมล้ำอยู่ที่ไหน? (Where is inequality?)**

เพื่อแสดงให้เห็นว่ารายได้แยกตามจังหวัดและแสดงถึงความเลื่อมล้ำอย่างชัดเจน จึงใช้ข้อมูลที่ไม่ได้ตัด outliers ออก สำหรับแนวคิดของ Visualization (The Policy Matrix) การสร้าง Scatter Plot ที่แกน X คือ "รายได้เฉลี่ยรวมของจังหวัด" และแกน Y คือ "ช่องว่างรายได้ (กลุ่มอาชีพที่รวยสุด - จนสุด ในจังหวัดนั้น)"

*   บนซ้าย (จนแต่เหลื่อมล้ำสูง): จังหวัดที่คนส่วนใหญ่จน แต่มีเศรษฐีเฉพาะกลุ่ม (ควรเน้นกระจายรายได้และลดช่องว่าง)
*   ล่างซ้าย (จนแบบเท่าเทียม): จังหวัดที่ทุกคนรายได้ต่ำเหมือนกันหมด (ควรเน้นดึงเม็ดเงินลงทุนเพื่อ ยกระดับรายได้โดยรวม)
*   บนขวา (รวยแต่เหลื่อมล้ำสูง): จังหวัดเศรษฐกิจดี แต่รวยกระจุกจนกระจาย (ควรเน้นสวัสดิการให้คนรากหญ้า)
*   ล่างขวา (รวยแบบเท่าเทียม): พื้นที่อุดมคติ (ต้นแบบความสำเร็จ)

In [ ]:
# คำนวณตัวชี้วัดของแต่ละจังหวัด (Aggregations)
prov_stats = df_clean.groupby('province').agg(
    Avg_Income=('income_thb', 'mean'),   # รายได้เฉลี่ยรวมของจังหวัด (แกน X)
    Min_Income=('income_thb', 'min'),    # รายได้กลุ่มอาชีพที่จนที่สุด
    Max_Income=('income_thb', 'max'),    # รายได้กลุ่มอาชีพที่รวยที่สุด
).reset_index()

# สร้าง Feature ความเหลื่อมล้ำ (แกน Y) = ความต่างระหว่างกลุ่มรวยสุดและจนสุดในจังหวัดเดียวกัน
prov_stats['Income_Gap'] = prov_stats['Max_Income'] - prov_stats['Min_Income']

# หาค่ากึ่งกลาง (Median) เพื่อตีเส้นแบ่ง 4 Quadrant
x_mid = prov_stats['Avg_Income'].median()
y_mid = prov_stats['Income_Gap'].median()

# สร้าง Visualization (Quadrant Scatter Plot)
fig = px.scatter(
    prov_stats,
    x='Avg_Income',
    y='Income_Gap',
    text='province', # แปะชื่อจังหวัดไว้ที่จุด
    size='Max_Income', # ขนาดจุดบ่งบอกถึงเพดานรายได้ที่สูงที่สุด
    color='Income_Gap', # สีแดง=เหลื่อมล้ำสูง, น้ำเงิน=เหลื่อมล้ำต่ำ
    color_continuous_scale='Turbo',
    title='<b>Policy Matrix: แผนผังความเหลื่อมล้ำและรายได้ระดับจังหวัด (Quadrant Analysis)</b>',
    labels={
        'Avg_Income': 'รายได้เฉลี่ยของจังหวัด (ความมั่งคั่งโดยรวม) ->',
        'Income_Gap': 'ช่องว่างรายได้ (ความเหลื่อมล้ำ) ->'
    }
)

# ตีเส้นแบ่ง 4 โซน
fig.add_vline(x=x_mid, line_dash="dash", line_color="gray")
fig.add_hline(y=y_mid, line_dash="dash", line_color="gray")

# ปรับแต่ง Layout ให้สวยงาม
fig.update_traces(textposition='top center', textfont=dict(size=10))
fig.update_layout(
    height=800,
    font=dict(size=14),
    annotations=[
        dict(x=x_mid*0.8, y=y_mid*1.5, text="<b>จนแต่เหลื่อมล้ำสูง</b><br>(ต้องลดช่องว่างด่วน)", showarrow=False, font=dict(color="red")),
        dict(x=x_mid*1.2, y=y_mid*1.5, text="<b>รวยแต่เหลื่อมล้ำสูง</b><br>(รวยกระจุก จนกระจาย)", showarrow=False, font=dict(color="orange")),
        dict(x=x_mid*0.8, y=y_mid*0.5, text="<b>จนแบบเท่าเทียม</b><br>(ต้องยกระดับรายได้ทั้งจังหวัด)", showarrow=False, font=dict(color="blue")),
        dict(x=x_mid*1.2, y=y_mid*0.5, text="<b>รวยแบบเท่าเทียม</b><br>(จังหวัดต้นแบบอุดมคติ)", showarrow=False, font=dict(color="green"))
    ]
)

# แสดงผลกราฟ Interactive
fig.show()

# พิมพ์ผลสรุปจังหวัดที่สุดโต่ง
most_unequal = prov_stats.sort_values(by='Income_Gap', ascending=False).iloc[0]
print(f"🚨 จังหวัดที่มีความเหลื่อมล้ำ (ช่องว่างรายได้) สูงที่สุดคือ: {most_unequal['province']}")
print(f"   โดยมีกลุ่มที่รวยที่สุดเฉลี่ย {most_unequal['Max_Income']:,.0f} บาท และจนที่สุดเพียง {most_unequal['Min_Income']:,.0f} บาท")
print(f"   (ช่องว่างห่างกันถึง {most_unequal['Income_Gap']:,.0f} บาท)")

💡 **Insight**: จันทบุรี (Chanthaburi) คือ "เมืองที่ความเหลื่อมล้ำสูงที่สุด"

จะสังเกตเห็นว่าจุดของ "จันทบุรี" โดดลอยขึ้นไปบนสุดขอบกราฟด้านขวา ด้วยช่องว่างรายได้ที่ห่างกันถึง กว่า 1.6 แสนบาท ต่อเดือน! กลุ่มเจ้าของที่ดินเชิงพาณิชย์ (เช่น ล้งผลไม้ หรือ สวนทุเรียนส่งออก) ดึงเพดานรายได้พุ่งสูง ในขณะที่กลุ่มลูกจ้างเกษตรกรทั่วไปยังอยู่ในฐานราก จันทบุรีจึงเป็นพื้นที่ที่ควรเน้นนโยบายกระจายรายได้และสวัสดิการ (ลดช่องว่าง) มากกว่าการกระตุ้นเศรษฐกิจภาพรวมเพียงอย่างเดียว

💡 **Insight**: การแยกโซนนโยบาย (Policy Segmentation)

  โซนล่างซ้าย (Bottom-Left): เช่น ยโสธร หรือจังหวัดในภาคอีสานส่วนใหญ่ ที่จุดกระจุกตัวกัน เป็นพื้นที่ "จนแบบเท่าเทียม" การให้เงินอุดหนุนเฉพาะกลุ่มจะไม่ช่วยอะไร นโยบายควรเป็นการดึง Mega Project หรืออุตสาหกรรมขนาดใหญ่เข้าไปเพื่อยกระดับรายได้ของทั้งจังหวัด


**4.3 📊 รายได้มาจากไหน และใครพึ่งพาอะไร? (What is the income structure?)**

"โครงสร้างรายได้" และ "สัดส่วนการพึ่งพา" กราฟที่เหมาะสมในการแสดงข้อมูล คือ Stacked Bar Chart (กราฟแท่งสัดส่วน 100%) เพราะจะเป็นการปรับให้ทุกอาชีพมีแท่งยาวเท่ากัน (100%) ทำให้ง่ายในการเปรียบเทียบว่าใครพึ่งพาอะไรเป็นสัดส่วนกี่เปอร์เซ็นต์ โดยไม่ถูกบิดเบือนจากความรวยหรือความจนโดยรวม

ทำการแบ่งรายได้ออกเป็น 3 ส่วนหลัก คือ:

* รายได้ประจำ (เงินสด): มั่นคงที่สุด

* รายได้ประจำ (ไม่ใช่เงินสด): เช่น สวัสดิการ, อาหารที่เพาะปลูกเอง (เปราะบางต่อสภาวะแทรกซ้อน)

* รายได้ไม่ประจำ (Irregular): เปราะบางที่สุด เสี่ยงตกงานหรือขาดรายได้

In [ ]:
# กรองเฉพาะองค์ประกอบรายได้ 3 ประเภทหลัก (หลีกเลี่ยงการนับซ้ำกับยอดรวม)
components = ['รายได้ที่เป็นตัวเงิน', 'รายได้ที่ไม่เป็นตัวเงิน', 'รายได้ไม่ประจำ (ที่เป็นตัวเงิน)']
df_comp = df_clean[df_clean['source_income2'].isin(components)].copy()

# คำนวณรายได้เฉลี่ยของแต่ละประเภท แย่งตามกลุ่มอาชีพ (soc_eco_class1)
struct = df_comp.groupby(['soc_eco_class1', 'source_income2'])['income_thb'].mean().reset_index()

# คำนวณสัดส่วนเป็นเปอร์เซ็นต์ (100% Stack)
totals = struct.groupby('soc_eco_class1')['income_thb'].sum().reset_index().rename(columns={'income_thb': 'Total'})
struct = struct.merge(totals, on='soc_eco_class1')
struct['Percentage'] = (struct['income_thb'] / struct['Total']) * 100

# จัดเรียงข้อมูลให้กลุ่มที่พึ่งพา "รายได้ที่ไม่เป็นตัวเงิน" หรือ "รายได้ไม่ประจำ" อยู่ด้านบนสุด (เพื่อโชว์ความเสี่ยง)
# สร้างคอลัมน์จำลองเพื่อการจัดเรียง
sort_risk = struct[struct['source_income2'] != 'รายได้ที่เป็นตัวเงิน'].groupby('soc_eco_class1')['Percentage'].sum().reset_index()
sort_risk = sort_risk.sort_values(by='Percentage', ascending=True)
struct['soc_eco_class1'] = pd.Categorical(struct['soc_eco_class1'], categories=sort_risk['soc_eco_class1'], ordered=True)
struct = struct.sort_values('soc_eco_class1')

# สร้าง Visualization
fig = px.bar(
    struct,
    x='Percentage',
    y='soc_eco_class1',
    color='source_income2',
    orientation='h',
    text=struct['Percentage'].apply(lambda x: f'{x:.1f}%' if x > 2 else ''), # โชว์ตัวเลขเฉพาะก้อนที่ใหญ่กว่า 2%
    color_discrete_map={
        'รายได้ที่เป็นตัวเงิน': '#2E86C1',        # สีน้ำเงิน (มั่นคง)
        'รายได้ที่ไม่เป็นตัวเงิน': '#F39C12',      # สีส้ม (เตือนความเสี่ยง)
        'รายได้ไม่ประจำ (ที่เป็นตัวเงิน)': '#E74C3C' # สีแดง (อันตราย/เปราะบาง)
    },
    title='<b>Income Structure: โครงสร้างและระดับความเปราะบางของรายได้แยกตามกลุ่มอาชีพ</b>',
    labels={'Percentage': 'สัดส่วนรายได้ (%)', 'soc_eco_class1': 'กลุ่มอาชีพ', 'source_income2': 'ประเภทรายได้'}
)

fig.update_layout(
    font=dict(size=14),
    barmode='stack',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()

จากกราฟด้านบน สามารถวิเคราะห์ได้ดังนี้  

1. จะเห็นว่า 'กลุ่มลูกจ้าง' มีโครงสร้างรายได้ที่แข็งแรงที่สุด โดยพึ่งพารายได้ที่เป็นตัวเงิน (สีน้ำเงิน) เกือบ 90-95%

   💡 นโยบายสำหรับกลุ่มนี้จึงไม่ควรมุ่งไปที่การแจกเงินสด แต่ควรเป็นการลดหย่อนภาษี หรือการเพิ่มทักษะ (Upskill) เพื่อเลื่อนขั้นเงินเดือน

2. 'กลุ่มเกษตรกร' มีโครงสร้างรายได้ของ 'ผู้ถือครองทำการเกษตร' จะเห็นรายได้ที่ไม่เป็นตัวเงิน (แถบสีส้ม) หนาผิดปกติ เมื่อเทียบกับอาชีพอื่น นี่แปลว่ารายได้ส่วนใหญ่ของเขาคือ 'การผลิตเพื่อบริโภคเอง' (Subsistence) หรือการแลกเปลี่ยนสินค้า"  

   💡 ความเสี่ยงทางเศรษฐกิจ: หากเกิดภัยแล้ง น้ำท่วม หรือวิกฤตโลกร้อน กลุ่มนี้จะสูญเสียทั้ง 'แหล่งรายได้' และ 'แหล่งอาหาร' ไปพร้อมกันทันที โครงสร้างพวกเขาเปราะบางกว่าพนักงานบริษัทมหาศาล

3. กลุ่มที่รับความเสี่ยงจากรายได้ไม่ประจำ หรือ 'ผู้ไม่ได้ปฏิบัติงานเชิงเศรษฐกิจ'  
เราพบว่ากลุ่มฐานราก (เช่น รับจ้างทั่วไป หรือรับเงินช่วยเหลือ) มีการพึ่งพารายได้ไม่ประจำ (สีแดง) สูง  
  💡 นโยบายสำหรับกลุ่มนี้ต้องเป็น โครงข่ายความปลอดภัยทางสังคม (Social Safety Net: SSN) เช่น การประกันรายได้ขั้นต่ำ หรือกองทุนฉุกเฉินระดับชุมชน


🌵 Note: สังเกตได้ว่า 'รายได้ที่ไม่เป็นตัวเงิน' (สีเหลือง) สูงสุด 2 กลุ่ม ซึ่งสะท้อนปัญหา 2 มิติ คือ

**มิติแรก** 'ผู้ไม่ได้ปฏิบัติงานเชิงเศรษฐกิจ' มีสัดส่วนสีเหลืองสูงสุด ซึ่งสะท้อนถึงสังคมสูงวัยหรือกลุ่มเปราะบางที่ต้องพึ่งพาสวัสดิการและครอบครัว นโยบายสำหรับกลุ่มนี้คือรัฐต้องเข้าไปดูแลเรื่องสาธารณสุขและที่อยู่อาศัย

**มิติที่สอง** 'ผู้ถือครองทำการเกษตร' ซึ่งเป็นกลุ่มคนวัยทำงานแท้ๆ แต่กลับมีสัดส่วนรายได้ที่ไม่ใช่เงินสดสูงมาก นี่คือสัญญาณอันตรายว่าเกษตรกรไทยกำลังติดกับดัก การทำฟาร์มเพื่อยังชีพ ขาดสภาพคล่องและเงินสดหมุนเวียน นโยบายสำหรับกลุ่มนี้จึงไม่ใช่แค่แจกเงิน แต่ต้องเป็นการแปลงมูลค่าผลผลิตให้กลายเป็นเงินสด (Monetization) เช่น การหาตลาดรองรับ หรือการแปรรูปสินค้า

## 📝 สรุป Key Insights

* ความเหลื่อมล้ำที่รุนแรงที่สุดไม่ได้เกิดระหว่าง นายจ้างกับลูกจ้าง แต่เกิดขึ้น ภายในกลุ่มลูกจ้างด้วยกันเอง (Skill Gap) นอกจากนี้
* การเปิดเผยให้เห็นจุด Outliers ในกลุ่มเจ้าของธุรกิจ ยังช่วยยืนยันว่าโครงสร้างเศรษฐกิจมีกลุ่ม "เศรษฐีเฉพาะกลุ่ม (Ultra-wealthy)" ที่ดึงเพดานรายได้ให้ดูสูงเกินจริง

ข้อเสนอแนะนโยบาย: เน้นการ Upskill ลูกจ้างระดับล่าง (เช่น แรงงานเกษตร) เพื่อยกระดับฐานะ มากกว่าการใช้นโยบายแบบเหมารวม

---

* ปัญหาความเหลื่อมล้ำในแต่ละจังหวัดไม่เหมือนกัน ในจังหวัดจันทบุรี มีรายได้รวมสูงแต่ช่องว่างความเหลื่อมล้ำกว้างอย่างชัดเจน
* ในขณะที่จังหวัดส่วนใหญ่ในภาคอีสานกระจุกตัวอยู่ในโซน "รายได้ต่ำแบบเท่าเทียม"

ข้อเสนอแนะนโยบาย:
* พื้นที่เหลื่อมล้ำสูง (เช่น จันทบุรี) -> ต้องใช้นโยบาย "กระจายรายได้และสวัสดิการ (Wealth Distribution)"
* พื้นที่รายได้ต่ำแบบเท่าเทียม -> ต้องใช้นโยบาย "ดึงดูดการลงทุนเพื่อยกระดับรายได้โดยรวม"
---

* "วิกฤตสภาพคล่อง" ที่ซ่อนอยู่ในกลุ่มเกษตรกร แม้พวกเขาจะอยู่ในวัยทำงาน (Active Labor) แต่รายได้ส่วนใหญ่กลับเป็น "สิ่งที่ไม่ใช่ตัวเงิน" (ปลูกเพื่อกินเอง/แลกเปลี่ยน) ทำให้พวกเขาไม่มีเงินสดไปต่อยอดหรือใช้หนี้ ซึ่งต่างจากกลุ่มเปราะบาง (ผู้สูงอายุ/คนว่างงาน) ที่รับสวัสดิการโดยธรรมชาติอยู่แล้ว

ข้อเสนอแนะนโยบาย: นโยบายสำหรับเกษตรกรต้องไม่ใช่แค่การให้เงินกู้ แต่ต้องเป็นการ "เปลี่ยนผลผลิตให้เป็นกระแสเงินสด (Monetization)" เพื่อแก้ปัญหาขาดสภาพคล่องอย่างยั่งยืน

